# ARVOR Observation Converter

This notebook inspects the `obs_seq` file created by the DART ARVOR converter.  
The converter itself should be run from the terminal. This notebook focuses on reading, summarizing, and visualizing the output observations.

## Objectives

In this notebook you will learn how to:

- Read a DART `obs_seq` file using pyDARTdiags
- Inspect observation types and metadata
- Visualize observations on a map
- Examine the data profiles (in depth)
- Compute basic observation statistics

## 1. Import Python libraries

In [ ]:
import pydartdiags.obs_sequence.obs_sequence as obsq

import os
import cmocean
import numpy             as np
import pandas            as pd
import cartopy.crs       as ccrs
import cartopy.feature   as cfeature
import matplotlib.pyplot as plt
import matplotlib.dates  as mdates

from pathlib              import Path
from pydartdiags.stats    import stats
from pydartdiags.matplots import matplots as mp
from IPython.display      import Markdown, display

## 2. Define paths

In [ ]:
# Path to DART repo (directory) 
basedir = Path(f"/glade/derecho/scratch/{os.environ['USER']}/inacawo/DART_training")

# Path to the ARVOR converter
arvor_dir = basedir / 'observations' / 'obs_converters' / 'ARVOR' 

# Path to the obs_seq file
obs_seq_file = arvor_dir / 'work' / 'obs_seq.arvor'
print(f"obs_seq file: {obs_seq_file}")

# Make sure the obs_Seq file exists
assert obs_seq_file.exists(), 'obs_seq file not found'

## 3. Read the obs_seq file

In [ ]:
# Read the obs seq file into a DataFrame
obs = obsq.ObsSequence(obs_seq_file)

# Uncomment to inspect available methods/attributes
# help(obs)

## 4. Preview the observation table

In [ ]:
# Examine the file
print(f"DataFrame shape: {obs.df.shape}")
print('\n')

display(obs.df.head())

# To view everything
# obs.df
# obs.all_obs

## 5. Summarize the observation sequence

In [ ]:
print("*" * 16)
print("obs_seq SUMMARY:")
print("*" * 16)

print(f"\nNumber of observations : {len(obs.df)}")
print(f"Number of obs types    : {len(obs.types)}")

# Available observation types in the obs_seq file 
# Each type is associated with a DART idenitifier number
print("\nObservation types:")
for kind, name in obs.types.items():
    print(f"  {kind:3d} : {name}")

# Number of copies in the obs_seq 
# observation, QC, .. could be more especially after assimilation 
print("\nObservation copies:")
for i, name in enumerate(obs.copie_names):
    print(f"  {i:2d} : {name}")

# Number of QCs
print("\nQC copies:")
for i, name in enumerate(obs.qc_copie_names):
    print(f"  {i:2d} : {name}")

display(
    obs.df["type"]
    .value_counts()
    .rename_axis("Observation Type")
    .to_frame("Count")
)


## 6. Observation locations and Values by type 

In [ ]:
df = obs.df.copy()

# Define obs info as a list of tuples to use for plotting
plot_specs = [
    ("FLOAT_TEMPERATURE", "Temperature", "C"  , cmocean.cm.thermal),
    ("FLOAT_SALINITY"   , "Salinity"   , "psu", cmocean.cm.haline),
]

# Define the cartopy projection: standard. un-projected Equidistant 
# Cylindrical coordinate system (lons, lats mapped to 2D cartesian grid)
proj = ccrs.PlateCarree()

# We will plot a figure containing 2 different plots (maps)
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 7),
    sharey=True,
    subplot_kw={"projection": proj}
)

# Loop over figure specs:
# ax: current axis; obs_type: T, U, V; cmap: colormap style
for ax, (obs_type, title, label, cmap) in zip(axes, plot_specs):

    # subset dataframe by observation type
    # i.e., only get data for each obs type
    this_ob = df[df["type"] == obs_type]

    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.OCEAN, facecolor="aliceblue")
    ax.add_feature(cfeature.LAKES, facecolor="lightsteelblue")
    ax.add_feature(cfeature.LAND, zorder=0, linewidth=0.5, facecolor="whitesmoke")
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    sc = ax.scatter(
        this_ob["longitude"],
        this_ob["latitude"],
        c=this_ob["observation"],
        s=80,
        cmap=cmap,
        transform=proj,
        zorder=5,
    )

    plt.colorbar(sc, ax=ax, label=label, shrink=0.25)

    ax.set_extent([110, 140, -10, 0], crs=proj)

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.2)
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(f"{title} (N={len(this_ob)})")

plt.tight_layout()

display(Markdown("""
The maps below display the location of the <mark>**ARVOR Floats (2 of them) in the InaCAWO domain**</mark>. Temperature and Salinity
observations are collected."""))

plt.show()

## 7. Profiles

In [ ]:
# Create a drifter ID from location
this_ob = df[df["type"] == obs_type].copy()

this_ob["arvor_id"] = (this_ob["longitude"].round(2).astype(str) + "_" + this_ob["latitude"].round(2).astype(str))

# Sort IDs so numbering is reproducible
ids = sorted(this_ob["arvor_id"].unique())

id_map = {id_: f"ARVOR Float {i+1}" for i, id_ in enumerate(ids)}

fig, axes = plt.subplots(
    nrows=1,
    ncols=len(plot_specs),
    figsize=(10, 6),
    sharey=True,
)

for ax, (obs_type, title, ylabel, cmap) in zip(axes, plot_specs):

    this_ob = df[df["type"] == obs_type].copy()

    this_ob["arvor_id"] = (this_ob["longitude"].round(2).astype(str) + "_" + this_ob["latitude"].round(2).astype(str))

    ids = sorted(this_ob["arvor_id"].unique())
    id_map = {id_: f"ARVOR Float {i+1}" for i, id_ in enumerate(ids)}

    for arvor_id in ids:

        prof = this_ob[this_ob["arvor_id"] == arvor_id].sort_values("vertical")
    
        lon0 = prof["longitude"].iloc[0]
        lat0 = prof["latitude"].iloc[0]
        
        ax.plot(prof["observation"], prof["vertical"], "-o", ms=4, lw=1.8,
                label=f"{id_map[arvor_id]} ({lon0:.2f}°E, {lat0:.2f}°N)")

        ax.invert_yaxis()
        ax.set_ylim(10000, 1)
        ax.set_yscale('log')
        
        ax.set_xlabel(ylabel, fontsize= 14)
        ax.set_ylabel('Depth (m)', fontsize= 14)
        ax.set_title(f"{title} (N={len(this_ob)})", fontsize=16)
        
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=12)

plt.tight_layout()

display(Markdown("""
Visualize the Temperature and Salinity at different depths from the 2 ARVOR Floats.  
""")) 

plt.show()

## 8. Observations Statistics

In [ ]:
print("Observation time range:")
print(df["time"].min())
print(df["time"].max())

print("\nLongitude range:")
print(f"{df['longitude'].min():.2f} {df['longitude'].max():.2f}")

print("\nLatitude range:")
print(f"{df['latitude'].min():.2f} {df['latitude'].max():.2f}")

print("\n")

summary = []

for obs_type, title, units, _ in plot_specs:

    this_ob = df[df["type"] == obs_type]

    summary.append({
        "Type" : title,
        "Count": len(this_ob),
        "Min"  : this_ob["observation"].min(),
        "Max"  : this_ob["observation"].max(),
        "Mean" : this_ob["observation"].mean(),
        "Std"  : this_ob["observation"].std(),
        "Error SD": np.sqrt(this_ob["obs_err_var"]).mean(),
    })

display(pd.DataFrame(summary))

## Key Takeaways

- The ARVOR converter creates temperature and salinity observations.
- Observations are stored in DART `obs_seq` format.
- pyDARTdiags can be used to inspect and visualize the observations.
- The floats deliver in-situ data at different depths, down to about 2 km.
- Observation values, locations, and errors should always be checked before assimilation.